# 📊 Sunum Grafikleri Üretici — Slide 7, 8, 9

**Üretilen dosyalar:**

| Dosya | Slide | Açıklama |
|-------|-------|----------|
| `slide7_training_strategy_mcldnn.png` | 7 (sol) | AWGN base vs Scratch vs Fine-Tuned (Rayleigh) |
| `slide7_training_strategy_petcgdnn.png` | 7 (sağ) | Aynı PET-CGDNN için |
| `slide8_model_comp_f1_rayleigh.png` | 8 (sol) | MCLDNN vs PET-CGDNN F1 on Rayleigh |
| `slide8_model_comp_mcc_rayleigh.png` | 8 (sağ) | MCLDNN vs PET-CGDNN MCC on Rayleigh |
| `slide9_confusion_matrix_comparison.png` | 9 | AWGN vs Rayleigh CM (SNR=+18dB) |

In [ ]:
# 1. Ortam Kurulumu
from google.colab import drive
drive.mount('/content/drive')

import os, sys, pickle
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'figure.facecolor': 'white'})

PROJECT_DIR = '/content/drive/MyDrive/AMR-Project'
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
FT_DIR = os.path.join(PROJECT_DIR, 'fine_tuning_results')
SAVE_DIR = os.path.join(PROJECT_DIR, 'Sunum_Grafikleri')
os.makedirs(SAVE_DIR, exist_ok=True)

print(f'Scratch sonuçları: {RESULTS_DIR}')
print(f'Fine-tune sonuçları: {FT_DIR}')
print(f'Çıktı dizini: {SAVE_DIR}')

In [ ]:
# 2. Veri Yükleme — results/ ve fine_tuning_results/ AYRI AYRI

def find_metric(base_dir, model, channel, metric='acc.pkl'):
    """Case-insensitive arama (K3/k3, K10/k10 farkını halleder)."""
    ch_variants = [channel]
    if 'k3' in channel.lower():
        ch_variants = ['rician_k3', 'rician_K3']
    elif 'k10' in channel.lower():
        ch_variants = ['rician_k10', 'rician_K10']
    for ch in ch_variants:
        tag = f'{model}_{ch}'
        for sub in ['predictions', '.']:
            p = os.path.join(base_dir, tag, sub, metric) if sub != '.' else os.path.join(base_dir, tag, metric)
            if os.path.exists(p):
                return pickle.load(open(p, 'rb'))
    return None

models = ['mcldnn', 'petcgdnn']
channels = ['awgn', 'rayleigh', 'rician_k3', 'rician_k10']
metric_files = {'acc': 'acc.pkl', 'f1': 'f1_macro_scores.pkl', 'mcc': 'mcc_metric_scores.pkl'}

# scratch = results/ dizininden, ft = fine_tuning_results/ dizininden
scratch = {}
ft = {}
for m in models:
    scratch[m] = {}
    ft[m] = {}
    for ch in channels:
        scratch[m][ch] = {mk: find_metric(RESULTS_DIR, m, ch, mf) for mk, mf in metric_files.items()}
        ft[m][ch] = {mk: find_metric(FT_DIR, m, ch, mf) for mk, mf in metric_files.items()}

# Durum raporu
model_titles = {'mcldnn': 'MCLDNN', 'petcgdnn': 'PET-CGDNN'}
ch_labels = {'awgn': 'AWGN', 'rayleigh': 'Rayleigh', 'rician_k3': 'Rician K=3', 'rician_k10': 'Rician K=10'}

print('📋 SCRATCH (results/) — acc / f1 / mcc:')
for m in models:
    for ch in channels:
        s = scratch[m][ch]
        a, f, c = '✅' if s['acc'] else '❌', '✅' if s['f1'] else '❌', '✅' if s['mcc'] else '❌'
        print(f'  {model_titles[m]:>10} / {ch_labels[ch]:<12}: acc={a} f1={f} mcc={c}')

print('\n📋 FINE-TUNED (fine_tuning_results/) — acc / f1 / mcc:')
for m in models:
    for ch in channels:
        s = ft[m][ch]
        a, f, c = '✅' if s['acc'] else '❌', '✅' if s['f1'] else '❌', '✅' if s['mcc'] else '❌'
        print(f'  {model_titles[m]:>10} / {ch_labels[ch]:<12}: acc={a} f1={f} mcc={c}')

---
## 3. SLIDE 7 — Training Strategy Comparison (Rayleigh)

Her model için 3 çizgi: AWGN Baseline vs Scratch-Trained vs Fine-Tuned

**Çıktı:** `slide7_training_strategy_mcldnn.png`, `slide7_training_strategy_petcgdnn.png`

In [ ]:
TARGET_CH = 'rayleigh'

for m in models:
    awgn_acc = scratch[m]['awgn']['acc']
    scratch_acc = scratch[m][TARGET_CH]['acc']
    ft_acc = ft[m][TARGET_CH]['acc']

    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    plotted = False

    if awgn_acc:
        snrs = sorted(awgn_acc.keys())
        ax.plot(snrs, [awgn_acc[s]*100 for s in snrs], 'o-', lw=2.5, ms=6,
                label='AWGN Baseline', color='#4CAF50')
        plotted = True
    if scratch_acc:
        snrs = sorted(scratch_acc.keys())
        ax.plot(snrs, [scratch_acc[s]*100 for s in snrs], 's--', lw=2.5, ms=6,
                label='Scratch on Rayleigh', color='#FF9800')
        plotted = True
    if ft_acc:
        snrs = sorted(ft_acc.keys())
        ax.plot(snrs, [ft_acc[s]*100 for s in snrs], '^-', lw=2.5, ms=6,
                label='Fine-Tuned on Rayleigh', color='#E63946')
        plotted = True

    if not plotted:
        plt.close()
        print(f'❌ {model_titles[m]}: Yeterli veri yok')
        continue

    ax.set_xlabel('SNR (dB)', fontsize=13)
    ax.set_ylabel('Classification Accuracy (%)', fontsize=13)
    ax.set_title(f'{model_titles[m]} — Training Strategy Comparison (Rayleigh)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=11, loc='lower right')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_ylim([0, 105])
    plt.tight_layout()
    fname = f'slide7_training_strategy_{m}.png'
    fig.savefig(os.path.join(SAVE_DIR, fname), dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✅ Kaydedildi: {fname}\n')

---
## 4. SLIDE 8 — Model Comparison (F1 & MCC) on Rayleigh

Fine-tuned sonuçlardan MCLDNN vs PET-CGDNN.

**Çıktı:** `slide8_model_comp_f1_rayleigh.png`, `slide8_model_comp_mcc_rayleigh.png`

In [ ]:
# --- F1 Score ---
mcl_f1 = ft['mcldnn']['rayleigh']['f1']
pet_f1 = ft['petcgdnn']['rayleigh']['f1']

if mcl_f1 and pet_f1:
    snrs = sorted(mcl_f1.keys())
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    ax.plot(snrs, [mcl_f1[s] for s in snrs], 'o-', lw=2.5, label='MCLDNN', color='#E63946')
    ax.plot(snrs, [pet_f1[s] for s in snrs], 's-', lw=2.5, label='PET-CGDNN', color='#1D3557')
    ax.set_xlabel('SNR (dB)', fontsize=13)
    ax.set_ylabel('F1 Score (Macro)', fontsize=13)
    ax.set_title('Model F1 Score Comparison on Rayleigh (Fine-Tuned)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=12, loc='lower right')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_ylim([0, 1.05])
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, 'slide8_model_comp_f1_rayleigh.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print('✅ slide8_model_comp_f1_rayleigh.png')
else:
    print('❌ F1 verisi eksik!')

# --- MCC ---
mcl_mcc = ft['mcldnn']['rayleigh']['mcc']
pet_mcc = ft['petcgdnn']['rayleigh']['mcc']

if mcl_mcc and pet_mcc:
    snrs = sorted(mcl_mcc.keys())
    fig, ax = plt.subplots(figsize=(10, 6), dpi=150)
    ax.plot(snrs, [mcl_mcc[s] for s in snrs], 'o-', lw=2.5, label='MCLDNN', color='#E63946')
    ax.plot(snrs, [pet_mcc[s] for s in snrs], 's-', lw=2.5, label='PET-CGDNN', color='#1D3557')
    ax.set_xlabel('SNR (dB)', fontsize=13)
    ax.set_ylabel('MCC', fontsize=13)
    ax.set_title('Model MCC Comparison on Rayleigh (Fine-Tuned)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=12, loc='lower right')
    ax.grid(True, linestyle='--', alpha=0.5)
    ax.set_ylim([-0.1, 1.05])
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, 'slide8_model_comp_mcc_rayleigh.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print('✅ slide8_model_comp_mcc_rayleigh.png')
else:
    print('❌ MCC verisi eksik!')

---
## 5. Performans Özet Tablosu

Tüm model × kanal × strateji kombinasyonları.

In [ ]:
print('=' * 90)
print('📊 PERFORMANS ÖZET TABLOSU')
print('=' * 90)
print(f'{"Model":>10} | {"Channel":>12} | {"Strategy":>12} | {"Avg Acc%":>8} | {"Max Acc%":>8} | {"Avg F1":>7} | {"Avg MCC":>8}')
print('-' * 90)

for m in models:
    for ch in channels:
        for label, src in [('Scratch', scratch), ('Fine-Tuned', ft)]:
            acc = src[m][ch]['acc']
            f1 = src[m][ch]['f1']
            mcc = src[m][ch]['mcc']
            if acc is None:
                continue
            avg_a = np.mean(list(acc.values())) * 100
            max_a = np.max(list(acc.values())) * 100
            avg_f = f'{np.mean(list(f1.values())):.4f}' if f1 else 'N/A'
            avg_m = f'{np.mean(list(mcc.values())):.4f}' if mcc else 'N/A'
            print(f'{model_titles[m]:>10} | {ch_labels[ch]:>12} | {label:>12} | {avg_a:>7.2f}% | {max_a:>7.2f}% | {avg_f:>7} | {avg_m:>8}')

print('=' * 90)

---
## 6. SLIDE 9 — Confusion Matrix (AWGN vs Rayleigh)

AWGN-eğitimli MCLDNN → SNR=+18dB'de AWGN vs Rayleigh test.

**Ön koşul:** `amr_all_in_one_core_toolkit.py` + dataset pkl'leri + model ağırlıkları Drive'da.

**Çıktı:** `slide9_confusion_matrix_comparison.png`

In [ ]:
sys.path.insert(0, PROJECT_DIR)
from amr_all_in_one_core_toolkit import (
    load_data, prepare_data_mcldnn, MCLDNN, calculate_confusion_matrix
)
import tensorflow as tf
print(f'TF: {tf.__version__}, GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
AWGN_PKL = os.path.join(PROJECT_DIR, 'RML2016.10a_dict.pkl')
RAY_PKL  = os.path.join(PROJECT_DIR, 'RML2016.10a_rayleigh.pkl')

# MCLDNN AWGN ağırlıkları
WEIGHTS = None
for ext in ['best_weights.keras', 'best_weights.h5', 'best_weights.weights.h5']:
    w = os.path.join(RESULTS_DIR, 'mcldnn_awgn', ext)
    if os.path.exists(w):
        WEIGHTS = w
        break

print(f'AWGN dataset: {"✅" if os.path.exists(AWGN_PKL) else "❌"}')
print(f'Rayleigh dataset: {"✅" if os.path.exists(RAY_PKL) else "❌"}')
print(f'MCLDNN weights: {"✅ " + WEIGHTS if WEIGHTS else "❌"}')

In [ ]:
if WEIGHTS and os.path.exists(AWGN_PKL) and os.path.exists(RAY_PKL):
    (mods, snrs, lbl_a), _, _, (Xt_a, Yt_a), (_, _, tidx_a) = load_data(AWGN_PKL)
    (_, _, lbl_r), _, _, (Xt_r, Yt_r), (_, _, tidx_r) = load_data(RAY_PKL)

    tf.keras.backend.clear_session()
    model = MCLDNN(weights=None, classes=len(mods))
    model.load_weights(WEIGHTS)
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    TARGET_SNR = 18

    def get_cm_at_snr(X_test, Y_test, lbl, test_idx, target_snr):
        test_snrs = [lbl[i][1] for i in test_idx]
        mask = np.where(np.array(test_snrs) == target_snr)
        X_snr, Y_snr = X_test[mask], Y_test[mask]
        inp = prepare_data_mcldnn(X_snr, X_snr, X_snr)
        Y_hat = model.predict(inp['test'], batch_size=400)
        cm, cor, ncor = calculate_confusion_matrix(Y_snr, Y_hat, mods)
        return cm, cor / (cor + ncor)

    cm_awgn, acc_awgn = get_cm_at_snr(Xt_a, Yt_a, lbl_a, tidx_a, TARGET_SNR)
    cm_ray, acc_ray   = get_cm_at_snr(Xt_r, Yt_r, lbl_r, tidx_r, TARGET_SNR)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6), dpi=150)
    for ax, cm, title_str in [
        (ax1, cm_awgn, f'AWGN (Acc={acc_awgn*100:.1f}%)'),
        (ax2, cm_ray, f'Rayleigh (Acc={acc_ray*100:.1f}%)')
    ]:
        im = ax.imshow(cm * 100, cmap='Blues', vmin=0, vmax=100)
        ax.set_title(title_str, fontsize=13, fontweight='bold')
        ax.set_xticks(np.arange(len(mods)))
        ax.set_xticklabels(mods, rotation=90, fontsize=9)
        ax.set_yticks(np.arange(len(mods)))
        ax.set_yticklabels(mods, fontsize=9)
        for i in range(len(mods)):
            for j in range(len(mods)):
                val = int(np.around(cm[i,j]*100))
                ax.text(j, i, val, ha='center', va='center', fontsize=7,
                        color='darkorange' if i==j else 'black')
        fig.colorbar(im, ax=ax, shrink=0.8)

    fig.suptitle(f'MCLDNN Confusion Matrix — SNR = +{TARGET_SNR} dB', fontsize=15, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(os.path.join(SAVE_DIR, 'slide9_confusion_matrix_comparison.png'), dpi=300, bbox_inches='tight')
    plt.show()
    print('✅ slide9_confusion_matrix_comparison.png')
else:
    print('❌ Gerekli dosyalar eksik!')

---
## 7. Üretilen Dosyalar

In [ ]:
print('\n' + '=' * 60)
print('📁 ÜRETİLEN DOSYALAR')
print('=' * 60)

expected = [
    'slide7_training_strategy_mcldnn.png',
    'slide7_training_strategy_petcgdnn.png',
    'slide8_model_comp_f1_rayleigh.png',
    'slide8_model_comp_mcc_rayleigh.png',
    'slide9_confusion_matrix_comparison.png',
]

for f in expected:
    path = os.path.join(SAVE_DIR, f)
    if os.path.exists(path):
        print(f'  ✅ {f} ({os.path.getsize(path)/1024:.0f} KB)')
    else:
        print(f'  ❌ {f}')

print(f'\n📌 Drive > AMR-Project > Sunum_Grafikleri')